# 延後初始化
:label:`sec_deferred_init`

到目前為止，我們忽略了建立網路時需要做的以下這些事情：

* 我們定義了網路架構，但沒有指定輸入維度。
* 我們添加層時沒有指定前一層的輸出維度。
* 我們在初始化參數時，甚至沒有足夠的資訊來確定模型應該包含多少參數。

有些讀者可能會對我們的程式碼能運行感到驚訝。
畢竟，深度學習框架無法判斷網路的輸入維度是什麼。
這裡的訣竅是框架的*延後初始化*（defers initialization），
即直到資料第一次通過模型傳遞時，框架才會動態地推斷出每個層的大小。

在以後，當使用卷積神經網路時，
由於輸入維度（即圖像的分辨率）將影響每個後續層的維數，
有了該技術將更加方便。
現在我們在寫程式碼時無須知道維度是什麼就可以設定參數，
這種能力可以大大簡化定義和修改模型的任務。
接下來，我們將更深入地研究初始化機制。

## 實例化網路

首先，讓我們實例化一個多層感知機。


此時，因為輸入維數是未知的，所以網路不可能知道輸入層權重的維數。
因此，框架尚未初始化任何參數，我們通過嘗試訪問以下參數進行確認。


接下來讓我們將資料通過網路，最終使框架初始化參數。


一旦我們知道輸入維數是20，框架可以通過代入值20來識別第一層權重矩陣的形狀。
識別出第一層的形狀後，框架處理第二層，依此類推，直到所有形狀都已知為止。
注意，在這種情況下，只有第一層需要延後初始化，但是框架仍是按順序初始化的。
等到知道了所有的參數形狀，框架就可以初始化參數。

## 小結

* 延後初始化使框架能夠自動推斷參數形狀，使修改模型架構變得容易，避免了一些常見的錯誤。
* 我們可以通過模型傳遞資料，使框架最終初始化參數。

## 練習

1. 如果指定了第一層的輸入尺寸，但沒有指定後續層的尺寸，會發生什麼？是否立即進行初始化？
1. 如果指定了不匹配的維度會發生什麼？
1. 如果輸入具有不同的維度，需要做什麼？提示：查看參數繫結的相關內容。


[Discussions](https://discuss.d2l.ai/t/5770)


練習一：

1. 如果指定了第一層的輸入尺寸，但沒有指定後續層的尺寸，會發生什麼？是否立即進行初始化？

我的回答:



如果只指定第一層的輸入尺寸但不指定後續層的尺寸，會發生延後初始化（deferred initialization）。讓我們通過例子來說明：

```python
import torch
import torch.nn as nn

# 創建一個網路，只指定第一層的輸入維度
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(256, 10)
)

# 檢查初始化狀態
print("初始化前：")
for name, param in net.named_parameters():
    print(f"{name} shape: {param.shape}")

# 傳入數據才會完成初始化
X = torch.randn(2, 20)
y = net(X)

print("\n數據傳入後：")
for name, param in net.named_parameters():
    print(f"{name} shape: {param.shape}")
```

主要特點：
1. 框架會延遲初始化，直到第一次前向傳播
2. 這種機制的好處是：
   - 自動推斷層的形狀
   - 避免手動計算每層維度
   - 減少錯誤可能性
3. 但要注意：
   - 在第一次前向傳播前，無法訪問參數
   - 需要確保輸入維度正確

這種延後初始化機制使得神經網絡的構建更加靈活和方便。


練習二：

2. 如果指定了不匹配的維度會發生什麼？

我的回答:



如果指定了不匹配的維度，會在運行時引發錯誤。讓我們通過例子來演示：

````python
import torch
import torch.nn as nn

# 創建一個維度不匹配的網路
net = nn.Sequential(
    nn.Linear(20, 256),
    nn.ReLU(),
    nn.Linear(512, 10)  # 錯誤：輸入維度應該是256而不是512
)

try:
    # 嘗試運行
    X = torch.randn(2, 20)
    y = net(X)
except RuntimeError as e:
    print("錯誤信息：")
    print(e)
````

會發生：
1. RuntimeError：維度不匹配錯誤
2. 錯誤發生在第一次前向傳播時
3. PyTorch會給出具體的錯誤信息，指出：
   - 預期的維度
   - 實際的維度
   - 錯誤發生的位置

這種錯誤檢查機制有助於：
- 及早發現設計錯誤
- 幫助調試
- 確保網路結構的正確性


練習三：

1. 如果輸入具有不同的維度，需要做什麼？提示：查看參數繫結的相關內容。

我的回答:





如果需要處理不同維度的輸入，有幾種解決方案：

1. 使用自適應層：
`````python
import torch
import torch.nn as nn

class FlexibleNet(nn.Module):
    def __init__(self):
        super().__init__()
        # 自適應池化層可以處理不同大小的輸入
        self.adaptive_pool = nn.AdaptiveAvgPool1d(256)
        self.fc = nn.Linear(256, 10)
    
    def forward(self, x):
        # 確保輸入是3D的 (batch_size, channels, length)
        if x.dim() == 2:
            x = x.unsqueeze(1)
        x = self.adaptive_pool(x)
        # 展平
        x = x.view(x.size(0), -1)
        return self.fc(x)

# 測試不同維度的輸入
net = FlexibleNet()
x1 = torch.randn(32, 100)    # 維度1
x2 = torch.randn(32, 200)    # 維度2
y1 = net(x1)
y2 = net(x2)
print(f"輸出1形狀: {y1.shape}")
print(f"輸出2形狀: {y2.shape}")
`````


2. 使用動態繫結：
`````python
class DynamicNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = None
    
    def forward(self, x):
        # 第一次運行時初始化層
        if self.fc is None:
            self.fc = nn.Linear(x.shape[1], 10)
        return self.fc(x)
`````


3. 預處理方法：
`````python
def preprocess(x, target_size=100):
    # 插值到固定大小
    if x.shape[1] != target_size:
        x = torch.nn.functional.interpolate(
            x.unsqueeze(1), 
            size=target_size, 
            mode='linear'
        ).squeeze(1)
    return x
`````


主要處理策略：
1. 使用自適應層自動調整
2. 動態創建適合輸入維度的層
3. 將輸入預處理到固定維度
4. 使用填充或裁剪到固定大小

選擇哪種方法取決於：
- 具體應用場景
- 性能要求
- 精度要求
- 計算資源限制
